![image.png](../images/image-tilelang_ascend_compile_run.png)

# 工程实践：编译、集成与部署

### 内容简介
- 回顾算子编译的 AOT 模式 & JIT 模式
- TileLang-Ascend 算子集成至 PyTorch

In [18]:
!pwd

/mnt/workspace/tilelang-ascend/docs/notebook


### AOT（Ahead-Of-Time）模式

- 提前编译好，先编译，后运行

In [1]:
import tilelang
import tilelang.language as T
import torch

/mnt/workspace/tilelang-ascend/venv/lib/python3.11/site-packages/torch_npu/__init__.py:316: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default.                      Do not set it to 1 to prevent some unknown errors
  warnings.warn("On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. \


> 以 [examples/developer_mode/gemm_developer.py](../../examples/developer_mode/gemm_developer.py) 为例

In [2]:
# 这里不用 @jit
# @tilelang.jit(out_idx=[-1], pass_configs=pass_configs)
def matmul(M, N, K, block_M, block_N, K_L1, dtype="float16", accum_dtype="float"):
    m_num = M // block_M
    n_num = N // block_N

    @T.prim_func
    def main(
        A: T.Tensor((M, K), dtype),  # type: ignore
        B: T.Tensor((K, N), dtype),  # type: ignore
        C: T.Tensor((M, N), dtype),  # type: ignore
    ):
        with T.Kernel(m_num * n_num, is_npu=True) as (cid, _):
            bx = cid // n_num
            by = cid % n_num

            A_L1 = T.alloc_shared((block_M, K_L1), dtype)
            B_L1 = T.alloc_shared((K_L1, block_N), dtype)

            C_L0 = T.alloc_fragment((block_M, block_N), accum_dtype)

            loop_k = T.ceildiv(K, K_L1)
            for k in T.serial(loop_k):
                T.copy(A[bx * block_M, k * K_L1], A_L1)
                T.copy(B[k * K_L1, by * block_N], B_L1)

                T.gemm_v0(A_L1, B_L1, C_L0, init=(k == 0))


            T.copy(C_L0, C[bx * block_M, by * block_N])

    return main

In [3]:
M = 1024
N = 1024
K = 1024

In [4]:
func = matmul(M, N, K, 128, 256, 64)
type(func)  # T.prim_func 装饰后的函数

tvm.tir.function.PrimFunc

In [5]:
pass_configs = {
    tilelang.PassConfigKey.TL_ASCEND_AUTO_CV_COMBINE: True,
    tilelang.PassConfigKey.TL_ASCEND_AUTO_SYNC: True,
    tilelang.PassConfigKey.TL_ASCEND_MEMORY_PLANNING: True,
}

# 提供 pass_configs
with tilelang.transform.PassContext(opt_level=3, config=pass_configs):
    # 手动 lower
    kernel = tilelang.engine.lower(func)

kernel

CompiledArtifact(host_mod=None, device_mod=None, params=[KernelParam(dtype=torch.float16, shape=[1024, 1024]), KernelParam(dtype=torch.float16, shape=[1024, 1024]), KernelParam(dtype=torch.float16, shape=[1024, 1024])], kernel_source='#include "tl_templates/ascend/common.h"\n#include "acl/acl.h"\n#include <runtime/rt_ffts.h>\nusing namespace Catlass;\nusing uint = unsigned int;\nusing uchar = unsigned char;\nusing ushort = unsigned short;\n\nextern "C" __global__ __aicore__ void main_kernel( GM_ADDR A_handle,  GM_ADDR B_handle,  GM_ADDR C_handle, uint64_t fftsAddr) {\n  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_MIX_AIC_1_2);\n  AscendC::TPipe pipe;\n\n  AscendC::GlobalTensor<half> A;\n  A.SetGlobalBuffer((__gm__ half*)A_handle);\n  AscendC::GlobalTensor<half> B;\n  B.SetGlobalBuffer((__gm__ half*)B_handle);\n  AscendC::GlobalTensor<half> C;\n  C.SetGlobalBuffer((__gm__ half*)C_handle);\n\n  AscendC::TBuf<AscendC::TPosition::A2> ascend_l0a;\n  pipe.InitBuffer(ascend_l0a, 65536);\n  AscendC::

In [6]:
# 最终生成的 AscendC 源码
with open("example_gemm.cpp", "w") as f:
    f.write(kernel.kernel_source)

查看生成的源码：[example_gemm.cpp](./example_gemm.cpp)

In [7]:
%%bash
# 手动编译 .so
bisheng --npu-arch=dav-2201 \
-O2 \
-std=c++17 \
-xasc \
-I${ASCEND_HOME_PATH}/include \
-I${ASCEND_HOME_PATH}/include/experiment/msprof \
-I${ASCEND_HOME_PATH}/include/experiment/runtime \
-I${ASCEND_HOME_PATH}/pkg_inc \
-I${ASCEND_HOME_PATH}/pkg_inc/runtime \
-I${ASCEND_HOME_PATH}/pkg_inc/profiling \
-I${TL_ROOT}/3rdparty/catlass/include \
-I${TL_ROOT}/3rdparty/shmem/include \
-I${TL_ROOT}/3rdparty/shmem/src/device \
-I${TL_ROOT}/src \
-DBACKEND_HYBM \
-L${ASCEND_HOME_PATH}/lib64 \
-Wno-macro-redefined \
-Wno-ignored-attributes \
-Wno-non-c-typedef-for-linkage \
-lruntime \
-lascendcl \
-lm \
-ltiling_api \
-lplatform \
-lc_sec \
-ldl \
-fPIC \
--shared \
example_gemm.cpp \
-o libop.so

# 把 example_gemm.cpp 编译成 libop.so

> 编译指令参考 [tilelang/jit/adapter/libgen.py](../../tilelang/jit/adapter/libgen.py)

In [8]:
import ctypes

torch.manual_seed(0)
torch.set_default_device("npu")

In [9]:
a = torch.randn(M, K).half()
b = torch.randn(K, N).half()
c = torch.empty(M, N).half()

print("Init Successful!")

Init Successful!


In [10]:
# 加载 .so 并调用算子
lib = ctypes.CDLL("./libop.so")


def tl_gemm(A: torch.Tensor, B: torch.Tensor, C: torch.Tensor):
    stream = torch.npu.current_stream()._as_parameter_

    # 调用 void call(...)
    lib.call(
        ctypes.c_void_p(A.data_ptr()),
        ctypes.c_void_p(B.data_ptr()),
        ctypes.c_void_p(C.data_ptr()),
        stream
    )

tl_gemm(a, b, c)

torch.npu.synchronize()

ref_c = a @ b

torch.testing.assert_close(c, ref_c, rtol=1e-2, atol=1e-2)
print("Kernel Output Match!")

Kernel Output Match!


### JIT（Just-In-Time）模式

- 运行中即时编译

In [11]:
# 一如既往地使用 @tilelang.jit
@tilelang.jit(out_idx=[-1], pass_configs=pass_configs)
def matmul(M, N, K, block_M, block_N, K_L1, dtype="float16", accum_dtype="float"):
    m_num = M // block_M
    n_num = N // block_N

    @T.prim_func
    def main(
        A: T.Tensor((M, K), dtype),  # type: ignore
        B: T.Tensor((K, N), dtype),  # type: ignore
        C: T.Tensor((M, N), dtype),  # type: ignore
    ):
        with T.Kernel(m_num * n_num, is_npu=True) as (cid, _):
            bx = cid // n_num
            by = cid % n_num

            A_L1 = T.alloc_shared((block_M, K_L1), dtype)
            B_L1 = T.alloc_shared((K_L1, block_N), dtype)

            C_L0 = T.alloc_fragment((block_M, block_N), accum_dtype)

            loop_k = T.ceildiv(K, K_L1)
            for k in T.serial(loop_k):
                T.copy(A[bx * block_M, k * K_L1], A_L1)
                T.copy(B[k * K_L1, by * block_N], B_L1)

                T.gemm_v0(A_L1, B_L1, C_L0, init=(k == 0))


            T.copy(C_L0, C[bx * block_M, by * block_N])

    return main

In [12]:
# 不开启编译缓存，这样获取 kernel 函数时一定会触发编译，方便演示
tilelang.disable_cache()

In [13]:
# 编译得到 kernel 函数
kernel = matmul(M, N, K, 128, 256, 64)
type(kernel)

tilelang.jit.kernel.JITKernel

In [14]:
c = kernel(a, b)

ref_c = a @ b

torch.testing.assert_close(c, ref_c, rtol=1e-2, atol=1e-2)
print("Kernel Output Match!")

Kernel Output Match!


In [15]:
# 查看编译出的 .so 的路径
kernel.adapter.libpath

'/tmp/tmpdz4zw5s4.so'

In [16]:
# 删除旧的 libop.so
!rm ./libop.so
!ls

aot_jit.ipynb  example_gemm.cpp


In [17]:
# 复制新编译好的 libop.so 到当前目录
!cp {kernel.adapter.libpath} ./libop.so
!ls

aot_jit.ipynb  example_gemm.cpp  libop.so


### TileLang-Ascend 算子集成至 PyTorch

- [样例：flash_attention 算子集成至 PyTorch](../../examples/torch_tl_ascend/overview.ipynb)